In [2]:
import requests
import time
import os
import json
import pandas as pd
from IPython.display import clear_output

# Configurações
BUFFER_SIZE = 200  # número de apps antes de salvar no arquivo
DATA_DIR = "../data/raw"
BATCH_FILE_PREFIX = os.path.join(DATA_DIR, "games_batch")
CHECKPOINT_FILE = os.path.join(DATA_DIR, "checkpoint.csv")

# Buffers
game_data_buffer = []
processed_apps_buffer = []

# Sessão persistente
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

# --- Funções --- #

def ensure_data_dir():
    """Garante que o diretório data existe"""
    if not os.path.exists(DATA_DIR):
        os.makedirs(DATA_DIR)
        print(f"📁 Diretório criado: {DATA_DIR}")

def load_checkpoint():
    """Carrega lista de app_ids processados a partir do checkpoint CSV"""
    ensure_data_dir()
    if os.path.exists(CHECKPOINT_FILE):
        try:
            df = pd.read_csv(CHECKPOINT_FILE)
            processed = df['app_id'].tolist()
            print(f"📂 Checkpoint carregado: {len(processed)} apps processados anteriormente.")
            return processed
        except Exception as e:
            print(f"❌ Erro ao carregar checkpoint: {e}")
            return []
    return []

def save_buffers():
    """
    Salva buffers de jogos em Parquet e checkpoint em CSV.
    - Converte todas as colunas para string para evitar erros de tipagem.
    """
    global game_data_buffer, processed_apps_buffer

    # --- Salvar jogos em Parquet ---
    if game_data_buffer:
        try:
            df_games = pd.DataFrame(game_data_buffer)

            # Converter todas as colunas para string
            for col in df_games.columns:
                df_games[col] = df_games[col].apply(lambda x: json.dumps(x, ensure_ascii=False) 
                                                    if isinstance(x, (dict, list)) 
                                                    else str(x) if x is not None else "")

            timestamp = int(time.time())
            parquet_file = f"{BATCH_FILE_PREFIX}_{timestamp}.parquet"

            df_games.to_parquet(parquet_file, index=False, engine='pyarrow')
            print(f"💾 Batch salvo: {parquet_file} ({len(game_data_buffer)} registros)")

            game_data_buffer = []
        except Exception as e:
            print(f"❌ Erro ao salvar batch Parquet: {e}")
            raise e

    # --- Salvar checkpoint em CSV ---
    if processed_apps_buffer:
        try:
            df_checkpoint = pd.DataFrame({'app_id': [str(a) for a in processed_apps_buffer]})
            header = not os.path.exists(CHECKPOINT_FILE)
            df_checkpoint.to_csv(CHECKPOINT_FILE, mode='a', header=header, index=False)
            print(f"📝 Checkpoint atualizado: {len(processed_apps_buffer)} apps")
            processed_apps_buffer = []
        except Exception as e:
            print(f"❌ Erro ao salvar checkpoint CSV: {e}")

    clear_output(wait=True)

def save_game_data(game_data):
    """Adiciona app ao buffer de dados"""
    global game_data_buffer
    game_data_buffer.append(game_data)

def get_all_steam_apps():
    """Retorna lista de todos os app_ids da Steam"""
    url = "https://api.steampowered.com/ISteamApps/GetAppList/v2/"
    try:
        response = session.get(url)
        response.raise_for_status()
        data = response.json()
        return [app['appid'] for app in data['applist']['apps']]
    except Exception as e:
        print(f"❌ Erro ao buscar lista de apps: {e}")
        return []

def get_game_details(app_id, retries=10):
    """Obtém full_data de um jogo da Steam"""
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}&l=en&cc=US"
    
    for attempt in range(retries):
        try:
            response = session.get(url)
            response.raise_for_status()
            data = response.json()
            
            game_data = data.get(str(app_id), {})
            if game_data.get("success"):
                return game_data["data"]
            
            print(f"⚠️ AppID {app_id} não possui dados disponíveis.")
            return None
            
        except requests.exceptions.HTTPError as e:
            if response.status_code == 429:
                print(f"⏳ 429 Too Many Requests para AppID {app_id}, aguardando 100s...")
                time.sleep(100)
                continue

            if attempt == (retries - 1):
                raise e
            return None
        except requests.exceptions.RequestException as e:
            print(f"🚨 Erro de requisição para AppID {app_id}: {e}, aguardando 30s...")
            time.sleep(30)
            if attempt == (retries -1):
                raise e
            continue
    
    return None

# --- Main --- #

def main():
    ensure_data_dir()
    all_apps = get_all_steam_apps()
    if not all_apps:
        print("❌ Nenhum app encontrado. Encerrando.")
        return

    processed_apps = load_checkpoint()
    apps_to_process = [a for a in all_apps if a not in processed_apps]
    
    print(f"⚡ Total de apps a processar: {len(apps_to_process)}")
    print(f"📁 Dados serão salvos em: {DATA_DIR}")

    for i, app_id in enumerate(apps_to_process, start=1):
        print(f"🔎 Processando AppID {app_id} ({i}/{len(apps_to_process)})")
        
        details = get_game_details(app_id)
        
        # Sempre salva o app_id no checkpoint, mesmo que não exista
        processed_apps_buffer.append(app_id)
        
        if details:
            save_game_data(details)

        gd_count = len(game_data_buffer)
        print(f"📊 Progresso Buffer: {gd_count}/{BUFFER_SIZE}")
        if gd_count >= BUFFER_SIZE:
            save_buffers()

        time.sleep(2)  # evita sobrecarga da API

    # Salvar o que sobrou no buffer
    save_buffers()
    print(f"🎉 Extração concluída! Dados salvos em: {DATA_DIR}")

if __name__ == "__main__":
    main()

🎉 Extração concluída! Dados salvos em: ../data/raw


In [1]:
import pandas as pd
import os
import glob
import pyarrow as pa

pa.register_extension_type.__globals__["_registry"] = {}

def read_all_parquets(data_dir="../../next-level/data/raw"):
    parquet_files = glob.glob(os.path.join(data_dir, "games_batch_*.parquet"))
    print("🔎 Arquivos encontrados:", parquet_files)
    print("📂 Diretório atual:", os.getcwd())
    print(parquet_files)
    if not parquet_files:
        print("❌ Nenhum arquivo Parquet encontrado")
        return None
    
    dataframes = []
    for file in parquet_files:
        df = pd.read_parquet(file)
        dataframes.append(df)
        print(f"📖 Lido: {os.path.basename(file)} - {len(df)} registros")
    
    # Combinar todos os DataFrames
    combined_df = pd.concat(dataframes, ignore_index=True)
    print(f"\n🎯 Total combinado: {len(combined_df)} registros")
    return combined_df

df = read_all_parquets()


🔎 Arquivos encontrados: ['../../next-level/data/raw/games_batch_1760223083.parquet', '../../next-level/data/raw/games_batch_1759871862.parquet', '../../next-level/data/raw/games_batch_1759963184.parquet', '../../next-level/data/raw/games_batch_1759865827.parquet', '../../next-level/data/raw/games_batch_1760256882.parquet', '../../next-level/data/raw/games_batch_1760049491.parquet', '../../next-level/data/raw/games_batch_1759875950.parquet', '../../next-level/data/raw/games_batch_1759698421.parquet', '../../next-level/data/raw/games_batch_1760163356.parquet', '../../next-level/data/raw/games_batch_1759941080.parquet', '../../next-level/data/raw/games_batch_1760152762.parquet', '../../next-level/data/raw/games_batch_1760191803.parquet', '../../next-level/data/raw/games_batch_1759918583.parquet', '../../next-level/data/raw/games_batch_1760239549.parquet', '../../next-level/data/raw/games_batch_1759956492.parquet', '../../next-level/data/raw/games_batch_1759997572.parquet', '../../next-lev